In [1]:
import re
import pandas as pd
import numpy as np

df = pd.read_csv('../data/raw/bank_churn_raw.csv')
print('raw shape:', df.shape)

raw shape: (10300, 22)


In [2]:
LEAK_COLS = ['exit_survey_score', 'account_closed_date', 'churn_flag_internal']
df = df.drop(columns=LEAK_COLS, errors='ignore')
print('after leakage drop:', df.shape)

after leakage drop: (10300, 19)


In [ ]:
def parse_income(val):
    if isinstance(val, (int, float)):
        return float(val)
    if pd.isna(val):
        return np.nan

    s = str(val).strip().upper()
    s = re.sub(r'PHP|\$|,|\s', '', s)

    mult = 1
    if s.endswith('K'):
        s = s[:-1]
        mult = 1_000
    elif s.endswith('M'):
        s = s[:-1]
        mult = 1_000_000

    try:
        return float(s) * mult
    except ValueError:
        return np.nan

df['annual_income_cleaned'] = df['annual_income'].apply(parse_income)
display(df[['annual_income', 'annual_income_cleaned']].sample(15, random_state=42))
display(df['annual_income_cleaned'].describe())

In [ ]:
before = len(df)
dup_id = df['customer_id'].duplicated().sum()

df = df.sort_values('account_open_date').drop_duplicates(subset=['customer_id'], keep='first')

after = len(df)
print('duplicate customer_id rows removed:', before - after)
print('remaining duplicate customer_id:', df['customer_id'].duplicated().sum())
print('shape after dedup:', df.shape)

In [3]:
high_risk_cols = ['complaints_12mo', 'credit_score', 'gender']
for c in high_risk_cols:
    if c in df.columns:
        df[f'{c}_missing'] = df[c].isna().astype(int)

mcar_numeric = ['digital_engagement', 'debt_to_income', 'monthly_transactions', 'num_products']
mcar_categorical = ['region', 'account_type', 'has_credit_card']

for c in mcar_numeric:
    if c in df.columns:
        df[c] = df[c].fillna(df[c].median())

for c in mcar_categorical:
    if c in df.columns and df[c].isna().sum() > 0:
        df[c] = df[c].fillna(df[c].mode(dropna=True).iloc[0])

# high-risk numeric impute (flags preserved)
for c in ['complaints_12mo', 'credit_score']:
    if c in df.columns:
        df[c] = df[c].fillna(df[c].median())

# high-risk categorical
if 'gender' in df.columns and df['gender'].isna().sum() > 0:
    df['gender'] = df['gender'].fillna(df['gender'].mode(dropna=True).iloc[0])

nulls_left = df.isna().sum().sort_values(ascending=False)
display(nulls_left[nulls_left > 0])
print('final shape:', df.shape)

Series([], dtype: int64)

final shape: (10300, 22)


In [ ]:
out_path = '../data/interim/bank_churn_cleaned.csv'
df.to_csv(out_path, index=False)
print('saved:', out_path)